### Ingestión del archivo "person.json"

In [0]:
%run "../includes/configuration"


In [0]:
%run "../includes/common_functions"

In [0]:
dbutils.widgets.text("p_environment","")
v_environment = dbutils.widgets.get("p_environment")


###Paso 1 - Leer el archivo JSON usando "DataFrameReader" de Spark
####La documentación para los comandos de Spark la sacamos de https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/index.html

In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType

In [0]:
name_schema = StructType(fields = [
    StructField("forename", StringType(), True),
    StructField("surname", StringType(), True)
])








In [0]:
persons_schema = StructType(fields = [
    StructField("personId", StringType(), False),
    StructField("personName", name_schema, True)
])



In [0]:
#persons_df = spark.read.schema(persons_schema).json("abfss://bronze@moviee.dfs.core.windows.net/person.json")

persons_df = spark.read.schema(persons_schema).json(f"{bronze_folder_path}/person.json")

In [0]:
persons_df.printSchema()

In [0]:
display(persons_df)

### Paso 2 - Renombrar las columnas y añadir columnas nuevas

In [0]:
from pyspark.sql.functions import col, concat, current_timestamp, lit

#persons_with_columns_df = persons_df \
#                        .withColumnRenamed("personId", "person_id") \
#                        .withColumn("name", concat(col("personName.forename"), lit(" "), col("personName.surname"))) \
#                        .withColumn("ingestion_date", current_timestamp()) \
#                        .withColumn("environment", lit("production")) \
#                        

persons_with_columns_df = persons_df \
                        .withColumnRenamed("personId", "person_id") \
                        .withColumn("name", concat(col("personName.forename"), lit(" "), col("personName.surname")))

persons_with_columns_df = add_ingestion_date(persons_with_columns_df) \
                          .withColumn("environment", lit(v_environment))
                        



                        

In [0]:
display(persons_with_columns_df)

### Paso 3 - Eliminar las columnas no requeridas

In [0]:
persons_final_df = persons_with_columns_df.drop(col("personName"))


In [0]:
display(persons_final_df)

### Paso 4 - Escribir la salida en un formato "Parquet"

In [0]:
#persons_final_df.write.mode("overwrite").parquet("abfss://silver@moviee.dfs.core.windows.net/persons")

persons_final_df.write.mode("overwrite").parquet(f"{silver_folder_path}/persons")


In [0]:
#display(spark.read.parquet("abfss://silver@moviee.dfs.core.windows.net/persons"))

display(spark.read.parquet(f"{silver_folder_path}/persons"))

### Paso 5 - Escribir datos en una managed table en el contenedor silver

#### Esto es un cambio posterior en el notebook, pasaría a reemplazar al paso 4, ya que ahora no quiero más generar archivos de salida en la capa silver a partir del dataframe, sino meter esa info en una tabla administrada por spark, pero dejo igual lo del paso 4, ya que esto se crea dentro de la carpeta _unitystorage y no pisa lo del paso 4

In [0]:
persons_final_df.write.mode("overwrite").format("delta").saveAsTable("movie_silver.persons")

In [0]:
%sql
SELECT * FROM movie_silver.persons

In [0]:
dbutils.notebook.exit("Success")